# PHASE 4: Modeling (Regression & Clustering)

In [ ]:
setwd(dirname(rstudioapi::getActiveDocumentContext()$path))
library(dplyr)
library(readr)
library(ggplot2)
library(randomForest)

### 1. Phân cụm (K-Means)

In [ ]:
model_data <- read_csv('../../data/airbnb_model_ready.csv', show_col_types = FALSE)

features <- model_data %>% select(price, latitude, longitude, number_of_reviews) %>% na.omit()
scaled_features <- scale(features)

set.seed(42)
kmeans_result <- kmeans(scaled_features, centers = 4, nstart = 10)
features$cluster <- as.factor(kmeans_result$cluster)

ggplot(features, aes(x = longitude, y = latitude, color = cluster)) +
  geom_point(alpha = 0.5, size = 0.5) +
  theme_minimal() +
  labs(title = "Phân khúc phòng Airbnb (K-Means Clustering)")

### 2. Dự đoán giá (Regression)

In [ ]:
set.seed(42)
df_reg <- model_data %>% 
  select(price, room_type, neighbourhood_group, accommodates, minimum_nights, has_reviews) %>%
  mutate(across(c(room_type, neighbourhood_group), as.factor)) %>%
  sample_n(10000)

train_idx <- sample(1:nrow(df_reg), 0.8 * nrow(df_reg))
train_data <- df_reg[train_idx, ]
test_data <- df_reg[-train_idx, ]

rf_model <- randomForest(price ~ ., data = train_data, ntree = 50, importance = TRUE)
predictions <- predict(rf_model, test_data)

mae <- mean(abs(test_data$price - predictions))
cat("Mean Absolute Error (MAE):", mae, "\n")

rss <- sum((predictions - test_data$price) ^ 2)
tss <- sum((test_data$price - mean(test_data$price)) ^ 2)
cat("R-squared (R2):", 1 - rss/tss, "\n")